# Peak assignments: the round trip

This notebook takes a sample's peaks **out** of Mascope, assigns them with an
engine of your own, **publishes** the result back as an assignment run, reads it
back, and compares it with Mascope's own engine on the same sample:

1. `samples.get_peaks` - the peaks, with the library lines they matched
2. a small external computation - one composition per peak
3. `peak_assignments.import_run` - the result published as a run of its own
4. `peak_assignments.get` - the run read back, as the app shows it
5. Mascope's own run on the sample, launched here if there is none
6. the two runs side by side, joined on `sample_peak_id`

A published run lives beside Mascope's: the same ledger, the run selector of the
Sample view, the batch ledger. It is always stamped with the engine that made
it. The rules of the import are on the page
[Publishing an assignment run](https://github.com/ultra-trace-systems/mascope/blob/master/docs/user/sdk/publishing-assignments.md).

> **Prerequisites:** the **editor** role on the sample, and a server with peak
> assignment switched on (`peak_assignment = true` under `[meta]`). Without
> either, the import is refused with a `PermissionDeniedError`.

In [ ]:
import numpy as np
import pandas as pd

from mascope_sdk import MascopeClient


mascope = MascopeClient(workspace="My Workspace")

### Pick a sample

Any sample with peaks will do; a blank has none and is refused.

In [ ]:
# Adjust this to match your data
batch = "My Batch"

samples = mascope.samples.list(batch=batch)
sample_id = samples.iloc[0]["sample_item_id"]
sample = mascope.samples.get(sample_id)
print(sample["sample_item_name"], sample["instrument_type"])

### 1. Peaks out

`get_peaks` returns every peak of the sample, one row per library line it
matched (a peak no line matched has one row with empty match columns).

An import states each peak's intensity, and it must be the quantity Mascope's
own engine uses: **peak height for an Orbitrap file, peak area for a TOF
file**. It weighs the sample's vote in the batch ledger, and nothing can tell
the wrong one apart.

In [ ]:
peaks = mascope.samples.get_peaks(sample_id)
INTENSITY = "height" if sample["instrument_type"] == "orbi" else "area"

print(
    f"{peaks['peak_id'].nunique()} peaks, {peaks['target_ion_id'].notna().sum()} library matches"
)

### 2. An engine of your own

The engine here is deliberately small: each peak takes the library line that
matched it best. It stands in for whatever you run outside Mascope - a research
engine, a script, a model. What matters is the frame it hands over: **one row
per peak**, in the columns of an import.

- `role` is `M0` for a peak that is its ion's monoisotopic line, and
  `iso_child` for another line of that ion, which names its monoisotopic peak
  in `owner_sample_peak_id`.
- `adduct` carries the notation (`[M-H]-`); `import_run` turns it into this
  server's mechanism id where exactly one mechanism has that notation.
- `engine_tier` is the engine's own verdict, by a rule of its own. The frame
  has **no `tier` column**: the server derives the tier from `fit_score`, the
  formula and the run's bands, which is the shape the import prefers.
- The engine's own numbers go in the row's `provenance` under
  `engine_provenance`: a `p_correct` at its top level would be stripped, as that
  name is the server's calibrated judgement.
- Every peak gets a row, the unexplained ones `unassigned`. A partial import
  is allowed, but the batch ledger takes the sample's whole contribution from
  its newest run, so a peak left out leaves the batch overview.

In [ ]:
ENGINE, ENGINE_VERSION = "best-line", "0.1.0"

# One reading per peak and one peak per library line: the best match wins.
lines = (
    peaks.dropna(subset=["target_isotope_id"])
    .sort_values("match_score_isotope", ascending=False)
    .drop_duplicates("peak_id")
    .drop_duplicates("target_isotope_id")
)
# A line that is its ion's own formula is the ion's monoisotopic peak; any
# other line is an isotopologue of that peak, or unexplained without one.
mono = lines["target_isotope_formula"] == lines["target_ion_formula"]
owner_of_ion = lines[mono].set_index("target_ion_id")["peak_id"]
owner = lines["target_ion_id"].map(owner_of_ion).where(~mono)
keep = mono | owner.notna()
lines, owner, mono = lines[keep], owner[keep], mono[keep]

assigned = pd.DataFrame(
    {
        "sample_peak_id": lines["peak_id"],
        "role": np.where(mono, "M0", "iso_child"),
        "owner_sample_peak_id": owner,
        "assigned_formula": lines["target_compound_formula"],
        "ion_formula": lines["target_ion_formula"],
        "isotope_formula": lines["target_isotope_formula"],
        "adduct": lines["ionization_mechanism"],
        "source": "database",
        "target_compound_id": lines["target_compound_id"],
        "target_ion_id": lines["target_ion_id"],
        "fit_score": lines["match_score_ion"].clip(0, 1),
        # The engine's own rule: a strong ion match is assigned, any other a candidate.
        "engine_tier": np.where(
            lines["match_score_ion"] >= 0.9, "assigned", "candidate"
        ),
        "provenance": [
            {
                "engine_provenance": {
                    "p_correct": round(float(score), 3),
                    "rule": "best line",
                }
            }
            for score in lines["match_score_isotope"]
        ],
    }
)

# Every peak of the sample, the unexplained ones unassigned.
rows = (
    peaks.drop_duplicates("peak_id")[["peak_id", "mz", INTENSITY]]
    .rename(
        columns={
            "peak_id": "sample_peak_id",
            "mz": "sample_peak_mz",
            INTENSITY: "sample_peak_intensity",
        }
    )
    .merge(assigned, on="sample_peak_id", how="left")
)
rows["role"] = rows["role"].fillna("unassigned")
rows["role"].value_counts()

### 3. Publish it

`import_run` checks the frame before anything is sent, and refuses what the
server would refuse anyway, naming the column and the rows: a value outside a
vocabulary, a peak twice, an owner that is not a row of the frame, a NaN where
no null is allowed. Then it sends the rows in as many requests as the server's
caps call for and returns the new run's id.

Two declarations go with the rows:

- `tier_bands`, the `assigned` and `candidate` thresholds the run tiers with,
  on the **evidence** scale: the fit weighted by the formula's chemical
  plausibility. These are the in-app engine's own, so the two runs' tiers mean
  the same.
- `calibration`, what the engine calibrated against. An import bypasses the
  sample's m/z verification, so this goes on the record in its place, and the
  run selector shows it.

In [ ]:
BANDS = {"assigned": 0.75, "candidate": 0.45}

run_id = mascope.peak_assignments.import_run(
    sample_id,
    rows,
    engine=ENGINE,
    engine_version=ENGINE_VERSION,
    tier_bands=BANDS,
    calibration={
        "method": "none",
        "note": "read the sample's calibrated m/z as served",
        "sample_mz_calibration_verified": (sample.get("mz_calibration") or {}).get(
            "verified"
        ),
    },
    config={"rule": "best library line per peak", "engine_tier_assigned_at": 0.9},
)
run_id

### 4. Read it back

The run is now the sample's latest completed run, so it is what `get()` and the
app show by default. Beside each row's `fit_score` the ledger carries the
`evidence` the server derived, and the `tier` it read off that evidence under
the run's bands. `engine_tier` is the engine's own verdict, untouched, and
`p_correct` is empty: that column is Mascope's calibrated judgement, which an
import may not write.

In [ ]:
theirs = mascope.peak_assignments.get(sample_id, run_id=run_id)

run = theirs.attrs["run"]
print(
    f"{run['engine']} {run['engine_version']}: {len(theirs)} rows, bands {run['tier_bands']}"
)
print(f"calibration: {run['calibration']}")
theirs[
    [
        "sample_peak_mz",
        "role",
        "assigned_formula",
        "fit_score",
        "evidence",
        "tier",
        "engine_tier",
        "p_correct",
    ]
].head(10)

Where the engine's verdict and the server's banding part, the ledger can say
so itself: `tier_disagrees=True` keeps only those rows, server-side.

In [ ]:
disputed = mascope.peak_assignments.get(sample_id, run_id=run_id, tier_disagrees=True)
print(
    f"{0 if disputed is None else len(disputed)} rows where the engine's tier is not the server's"
)
theirs.groupby(["engine_tier", "tier"], dropna=False).size().rename("rows")

### 5. Mascope's own run

The comparison needs a run of Mascope's engine on the same sample. The newest
completed one is used; when the sample has none, `assign` launches one and
waits for it, which takes a minute or a few.

In [ ]:
runs = mascope.peak_assignments.list_runs(sample_id)
in_app = runs[(runs["engine"] == "mascope") & (runs["status"] == "completed")]

if in_app.empty:
    ours = mascope.peak_assignments.assign(sample_id)
else:
    ours = mascope.peak_assignments.get(
        sample_id, run_id=in_app.iloc[0]["peak_assignment_run_id"]
    )
print(
    f"mascope {ours.attrs['run']['engine_version']}: {len(ours)} rows, bands {ours.attrs['run']['tier_bands']}"
)

### 6. Two engines, one sample

Both runs are rows of the same table, so a join on `sample_peak_id` puts them
side by side. The join is the one `tooling/assignment_compare/compare_runs.py`
in the Mascope repository makes, which does the same over whole batches: the
formulas compared regardless of element order, and each peak both engines
committed read as

- `same_formula` - the same neutral, the same adduct
- `same_neutral_other_adduct` - the same neutral through another adduct
- `same_ion_other_split` - the same ion read as another neutral and adduct,
  which no spectrum can tell apart
- `different_formula`

and a peak only one engine committed as `a_only_...` or `b_only_...`, naming
the other's role.

In [ ]:
import re


def canonical(formula):
    """An order-independent key for a formula, or None for none."""
    if not isinstance(formula, str) or not formula:
        return None
    counts = {}
    for element, n in re.findall(
        r"([A-Z][a-z]?)(\d*)", re.sub(r"[\[\]^]", "", formula)
    ):
        counts[element] = counts.get(element, 0) + int(n or 1)
    return "".join(f"{element}{counts[element]}" for element in sorted(counts))


notation = dict(
    zip(
        mascope.ionization.list()["ionization_mechanism_id"],
        mascope.ionization.list()["ionization_mechanism"],
    )
)


def side(ledger, prefix):
    return pd.DataFrame(
        {
            "sample_peak_id": ledger["sample_peak_id"],
            "mz": ledger["sample_peak_mz"],
            "intensity": ledger["sample_peak_intensity"],
            f"{prefix}_role": ledger["role"],
            f"{prefix}_formula": ledger["assigned_formula"].map(canonical),
            f"{prefix}_ion": ledger["ion_formula"].map(canonical),
            f"{prefix}_adduct": ledger["ionization_mechanism_id"].map(notation),
            f"{prefix}_fit": ledger["fit_score"],
            f"{prefix}_evidence": ledger["evidence"],
            f"{prefix}_tier": ledger["tier"],
            f"{prefix}_assignment": ledger["peak_assignment_id"],
        }
    )


def verdict(row):
    if row.a_role == "M0" and row.b_role == "M0":
        if row.a_formula == row.b_formula:
            same = (
                row.a_adduct == row.b_adduct
                or pd.isna(row.a_adduct)
                or pd.isna(row.b_adduct)
            )
            return "same_formula" if same else "same_neutral_other_adduct"
        if row.a_ion is not None and row.a_ion == row.b_ion:
            return "same_ion_other_split"
        return "different_formula"
    if row.a_role == "M0":
        return f"a_only_{row.b_role}"
    if row.b_role == "M0":
        return f"b_only_{row.a_role}"
    return f"{row.a_role}/{row.b_role}"


# a: Mascope's engine, b: the published one
joined = side(ours, "a").merge(
    side(theirs, "b").drop(columns=["mz", "intensity"]),
    on="sample_peak_id",
    how="outer",
)
for prefix in ("a", "b"):
    joined[f"{prefix}_role"] = joined[f"{prefix}_role"].fillna("missing")
joined["verdict"] = joined.apply(verdict, axis=1)
joined["verdict"].value_counts()

### Fit beside evidence, tiers under their own bands

`fit_score` is each engine's measurement as it reported it. `evidence` is the
fit weighted by the chemical plausibility of the formula, which the server
derives for both runs alike, and the quantity each run's tier was read off.
A tier is only comparable across runs under the bands that produced it, so
both are printed with their run's.

In [ ]:
both = joined[joined["verdict"] == "same_formula"]
print(
    f"Mascope bands {ours.attrs['run']['tier_bands']}, {ENGINE} bands {theirs.attrs['run']['tier_bands']}"
)
display(pd.crosstab(joined["a_tier"], joined["b_tier"], margins=True))
both[
    [
        "mz",
        "a_formula",
        "a_fit",
        "a_evidence",
        "b_fit",
        "b_evidence",
        "a_tier",
        "b_tier",
    ]
].sort_values("mz").head(15)

In [ ]:
import plotly.express as px
import plotly.io as pio


pio.templates.default = "plotly_dark"  # or "plotly_white"

committed = joined[joined["a_fit"].notna() & joined["b_fit"].notna()]
fig = px.scatter(
    committed,
    x="a_fit",
    y="b_fit",
    color="verdict",
    hover_data=["mz", "a_formula", "b_formula", "a_evidence", "b_evidence"],
    labels={"a_fit": "Mascope fit_score", "b_fit": f"{ENGINE} fit_score"},
    title="Fit scores of the peaks both engines committed",
)
fig.show()

### Two engines' confidence

Mascope's `p_correct` is its calibrated probability that an assignment is
correct, a column of the ledger. The published run's own figure is where the
engine put it, under `provenance.engine_provenance`, which `detail()` returns
with the row. Keeping both, under names that say whose they are, is what lets
the two be read side by side: here for the most intense peaks both engines
read alike.

In [ ]:
top = both.sort_values("intensity", ascending=False).head(10)
p_correct = ours.set_index("peak_assignment_id")["p_correct"]
confidence = pd.DataFrame(
    {
        "mz": top["mz"],
        "formula": top["a_formula"],
        "mascope_p_correct": top["a_assignment"].map(p_correct),
        f"{ENGINE}_p_correct": [
            (mascope.peak_assignments.detail(sample_id, pid)["provenance"] or {})
            .get("engine_provenance", {})
            .get("p_correct")
            for pid in top["b_assignment"]
        ],
    }
)
confidence

### Where to go from here

- `tooling/assignment_compare/compare_runs.py` makes this comparison over whole
  batches, with the agreement by intensity rank and a chemistry check of the
  committed formulas.
- In the app, the Sample view's run selector lists both runs, each with its
  engine; the published one carries its calibration disclosure.
- Runs are kept per engine: the nightly retention pass keeps an engine's
  newest runs per sample, so publishing again ages out this engine's older
  runs and never Mascope's own.
- An import that stopped partway leaves a run `importing`, which holds the
  sample until it ends: `peak_assignments.abandon_import(sample_id, run_id)`
  deletes it.